# RadLE OpenRouter quickstart

Run the RadLE v2 diagnostic prompt on your own ordered images. This notebook requests **one diagnosis**, not a correctness score. It makes no paid call until you set `RUN_LIVE = True` in the settings cell.

For now use `INPUT_MODE = "upload"`. The bundled case option is reserved for a later approved public case package; no clinical images are in this GitHub repository. Upasana can use the separately shared private image folder during development.


In [ ]:
%pip -q install openai==2.26.0 Pillow==12.1.1


## Set the model and image source

Use `INPUT_MODE = "upload"`. After upload, enter filenames in presentation order when prompted, or set `IMAGE_ORDER` before running. The bundled option will work only after an approved case package is released. Do not put the private Drive image folder, API key, or model outputs into GitHub.

`minimax_m3` is one configured example. Provider availability and study comparability must be checked before a paid run.


In [ ]:
INPUT_MODE = "upload"  # "upload" or "bundled"
MODEL_NAME = "muse_spark_1_3_openrouter"
BUNDLED_CASE_ID = "case_001"
BUNDLED_SOURCE = "github"  # "github" or "upload_zip"
IMAGE_ORDER = []  # optional: list filenames in presentation order before upload
RUN_LIVE = False  # explicitly change to True to spend API credit


In [ ]:
import base64, csv, hashlib, io, json, subprocess, time, zipfile
from pathlib import Path, PurePosixPath
from PIL import Image
from google.colab import files, userdata

MODELS = {
    "grok_4_6": ("x-ai/grok-4.6", "xai/zdr", "xhigh"),
    "claude_fable_5_1": ("anthropic/claude-fable-5.1", "anthropic", "max"),
    "deepseek_v4_flash_vision": ("deepseek/deepseek-v4-flash-vision-exp", "fireworks", "high"),
    "gemini_3_8_flash": ("google/gemini-3.8-flash", "google-ai-studio", "high"),
    "muse_spark_1_3_openrouter": ("meta/muse-spark-1.3", "meta", "xhigh"),
    "minimax_m3": ("minimax/minimax-m3", "minimax/fp8", None),
}
if MODEL_NAME not in MODELS:
    raise ValueError(f"Unsupported model: {MODEL_NAME}")

TEMPERATURES = {"minimax_m3": 0.01}


In [ ]:
PROMPT = 'System prompt:\n\nYou are a board-certified diagnostic radiologist. Given a medical image, provide the single most specific final diagnosis.\n\nRules:\n\t1.\tIf you can identify a diagnosis, return it. If you truly cannot, return “I don’t know”.\n\t2.\tNo abbreviations, use full words only.\n\t3.\tNo verbose descriptions of the diagnosis.\n\t4.\tUse the following Likert confidence scale (only when a diagnosis is given) to tell how confident you are in your diagnosis:\n\t∙\t0 = very low confidence (essentially a guess)\n\t∙\t1 = low confidence (weak leading diagnosis, several alternatives similarly plausible)\n\t∙\t2 = moderate confidence (one diagnosis favored, important alternatives remain)\n\t∙\t3 = high confidence (one diagnosis clearly favored, alternatives unlikely)\n\t∙\t4 = very high confidence (classic appearance, essentially certain)\n5. If you return “I don’t know”, then return Likert score as “null”.\n\nFinal output format:  Respond with this JSON:\n\n{"diagnosis": "<diagnosis in full words or I don\'t know>", "likert_score": <0-4 or null>}\n\n\nExample outputs:\n\n{"diagnosis": "Pulmonary tuberculosis", "likert_score": 3}\n\n{"diagnosis": "Von Hippel-Lindau syndrome", "likert_score": 4}\n\n{"diagnosis": "I don\'t know", "likert_score": null}'
PROMPT_SHA256 = hashlib.sha256(PROMPT.encode('utf-8')).hexdigest()
assert PROMPT_SHA256 == 'd409f2821f56deb57892f7bffaffd73234c9e5374e8cdfbcc864cc9cfbdbf3ba'
print(PROMPT)
print('Prompt SHA-256:', PROMPT_SHA256)


In [ ]:
if INPUT_MODE == "upload":
    uploaded = files.upload()
    if not IMAGE_ORDER:
        print("Uploaded filenames:", list(uploaded))
        IMAGE_ORDER = [name.strip() for name in input("Filenames in presentation order, comma-separated: ").split(",")]
    if set(IMAGE_ORDER) != set(uploaded) or len(IMAGE_ORDER) != len(uploaded):
        raise ValueError("IMAGE_ORDER must list every uploaded filename exactly once")
    image_paths = [Path(name) for name in IMAGE_ORDER]
    case_id = "uploaded_case"
elif INPUT_MODE == "bundled":
    repo = Path("/content/RadLE-Benchmark")
    if not repo.exists():
        if BUNDLED_SOURCE == "github":
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/DrHBSB/RadLE-Benchmark.git", str(repo)], check=True)
        elif BUNDLED_SOURCE == "upload_zip":
            uploaded_zip = files.upload()
            if len(uploaded_zip) != 1:
                raise ValueError("Upload exactly one RadLE-Benchmark ZIP archive")
            with zipfile.ZipFile(io.BytesIO(next(iter(uploaded_zip.values())))) as archive:
                for entry in archive.infolist():
                    parts = PurePosixPath(entry.filename).parts
                    if not parts or parts[0] != "RadLE-Benchmark" or ".." in parts:
                        raise ValueError("ZIP must contain only a RadLE-Benchmark top-level folder")
                    destination = Path("/content").joinpath(*parts)
                    if entry.is_dir():
                        destination.mkdir(parents=True, exist_ok=True)
                    else:
                        destination.parent.mkdir(parents=True, exist_ok=True)
                        destination.write_bytes(archive.read(entry))
        else:
            raise ValueError("BUNDLED_SOURCE must be 'github' or 'upload_zip'")
    with (repo / "data" / "radle_preprint_50" / "cases.csv").open(newline="", encoding="utf-8") as f:
        cases = {row["case_id"]: row for row in csv.DictReader(f)}
    if BUNDLED_CASE_ID not in cases:
        raise ValueError(f"Unknown bundled case: {BUNDLED_CASE_ID}")
    folder = repo / "data" / "radle_preprint_50" / "images" / BUNDLED_CASE_ID
    image_paths = sorted(folder.iterdir(), key=lambda p: p.name)
    if len(image_paths) != int(cases[BUNDLED_CASE_ID]["image_count"]):
        raise ValueError("Bundled image count mismatch")
    case_id = BUNDLED_CASE_ID
else:
    raise ValueError("INPUT_MODE must be 'upload' or 'bundled'")
print("Presentation order:", [p.name for p in image_paths])


In [ ]:
def image_payload(paths):
    content = [{"type": "text", "text": PROMPT}]
    for path in paths:
        raw = path.read_bytes()
        if raw.startswith(b"\x89PNG\r\n\x1a\n"):
            mime = "image/png"
        elif raw.startswith(b"\xff\xd8"):
            mime = "image/jpeg"
        else:
            raise ValueError(f"Unsupported image: {path.name}")
        with Image.open(path) as img:
            if img.format not in {"PNG", "JPEG"}:
                raise ValueError(f"Unsupported image: {path.name}")
            img.verify()
        content.append({"type": "image_url", "image_url": {"url": f"data:{mime};base64,{base64.b64encode(raw).decode('ascii')}"}})
    return content

CONTENT = image_payload(image_paths)
print(f"Validated {len(CONTENT) - 1} images. Model: {MODELS[MODEL_NAME][0]}")


In [ ]:
"""Parse the requested two-field response without judging correctness."""

from __future__ import annotations

import json
import re


def parse_response(raw_text):
    if not isinstance(raw_text, str) or not raw_text.strip():
        return "", None, "malformed_response"
    text = raw_text.strip()
    recovered = False
    fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.I | re.S)
    if fenced:
        text, recovered = fenced.group(1), True
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return "", None, "malformed_response"
    if not isinstance(data, dict):
        return "", None, "malformed_response"
    diagnosis = data.get("diagnosis")
    score = data.get("likert_score")
    if not isinstance(diagnosis, str) or not diagnosis.strip():
        return "", None, "malformed_response"
    diagnosis = diagnosis.strip()
    if diagnosis == "I don't know":
        if score is not None:
            return diagnosis, None, "invalid_abstention"
        return diagnosis, None, "abstained"
    if isinstance(score, bool) or not isinstance(score, int) or score not in range(5):
        return diagnosis, None, "invalid_likert"
    return diagnosis, score, "ok_recovered" if recovered else "ok"


In [ ]:
if RUN_LIVE:
    from openai import OpenAI
    api_key = userdata.get("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("Set OPENROUTER_API_KEY in Colab Secrets")
    model_id, provider_only, effort = MODELS[MODEL_NAME]
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
    started = time.perf_counter()
    try:
        kwargs = {
            "model": model_id,
            "messages": [{"role": "user", "content": CONTENT}],
            "max_tokens": 16384,
            "extra_body": {
                "response_format": {"type": "json_object"},
                "provider": {"only": [provider_only], "allow_fallbacks": False, "require_parameters": True},
            },
        }
        if effort is not None:
            kwargs["extra_body"]["reasoning"] = {"effort": effort, "exclude": False}
        if MODEL_NAME in TEMPERATURES:
            kwargs["temperature"] = TEMPERATURES[MODEL_NAME]
        response = client.chat.completions.create(**kwargs)
        raw = response.choices[0].message.content
        diagnosis, likert, status = parse_response(raw)
    except Exception as exc:
        diagnosis, likert, status = "", None, "api_error"
        print("API call failed:", type(exc).__name__)
    row = {"case_id": case_id, "model_id": model_id, "diagnosis": diagnosis,
           "likert_score": likert, "status": status,
           "latency_seconds": round(time.perf_counter() - started, 3),
           "prompt_sha256": PROMPT_SHA256}
    print(row)
else:
    row = None
    print("Ready. Set RUN_LIVE = True and rerun this cell to make one paid request.")


In [ ]:
if row is not None:
    output = Path("radle_prediction.csv")
    with output.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row))
        writer.writeheader()
        writer.writerow(row)
    files.download(str(output))


## Interpretation

The model's `likert_score` is its own 0–4 confidence label, not a calibrated probability. `abstained`, `malformed_response`, and `api_error` are kept distinct. This notebook does not know the reference diagnosis for an uploaded image and does not compute RadLE correctness or study scores.
